In [6]:
# Create a scenario object

class Scenario:
    def __init__(self, parameters_list, scenario_function):
        self.parameters_list = parameters_list
        self.scenario_function = scenario_function
                
    def run_scenario(self, cow, prev_output=None):
        # This will run phase_feed
        self.scenario_function(cow, *self.parameters_list, prev_output=prev_output)
        print(f"Finished scenario for cow {cow.name}")

In [7]:
# The scenario_function
def phase_feed(cow, param_1, param_2, prev_output=None):
    # First call without prev_output
    cow.execute_runModel(0,
                         50,
                         0.01,
                         7,
                         0,
                         scenario_param=param_1)

    # Get the prev_output from the first call
    prev_output_first_call = cow.results[-1]  # Assuming results is a list storing the dataframes

    # Second call with prev_output from the first call
    cow.execute_runModel(1,
                         100,  # Adjust the start time to continue from t=50
                         0.01,
                         7,
                         0,
                         scenario_param=param_2,
                         prev_output=prev_output_first_call)
    return


In [8]:
from simoolator.cow import Cow

class Herd:
    def __init__(self, outputs, model_functions, scenario_list):
        self.list_of_cows = []
        self.outputs = outputs 
        self.model_functions = model_functions
        self.scenario_list = scenario_list

    def add_cow(self, cow):
        self.list_of_cows.append(cow)        

    def rollcall(self):
        for cow in self.list_of_cows:
            cow.moo()
        print(f"There are {Cow.total_cows} cows in the herd!")
        return 
    
    def run_all_models(self,
                       Start,
                       runTime,
                       integInt,
                       communInt,
                       model_index,
                       prev_output=None,
                        output_file=False,
                        filepath='./',
                        filename='generic',
                        fileextension='.csv'
                    ):
        print("Begin executing models...")
        for cow in self.list_of_cows:
            cow.execute_runModel(Start,
                                runTime,
                                integInt,
                                communInt,
                                model_index,
                                prev_output,
                                output_file,
                                filepath,
                                filename,
                                fileextension
            )
        print("Finished")
        return
    
    def run_all_scenarios(self,
                          Start,
                        runTime,
                        integInt,
                        communInt,
                        model_index,
                        prev_output=None,
                        output_file=False,
                        filepath='./',
                        filename='generic',
                        fileextension='.csv'):
        # Will replace in future with Scenario class
        
        for cow in self.list_of_cows:
            for key, value in self.scenario_dict.items():
            # Eventually create a Scenario object with a method to pass variables this way
                scenario_param_name = key
                scenario_param_value = value

                cow.execute_runModel(Start, 
                                    runTime,
                                    integInt,
                                    communInt,
                                    model_index,
                                    scenario_param=scenario_param_value,
                                    prev_output=None,
                                    output_file=False,
                                    filepath='./',
                                    filename='generic',
                                    fileextension='.csv')
                print(f"Finished cow {cow.name} with {scenario_param_name}={scenario_param_value}")
        print("Finished all cows")
        return
    
    def import_dataframe(self, df):
        for index, row in df.iterrows():
            cow_variable = row['cow_ID']
  
            cow_variable = Cow(name=row['cow_ID'], 
                               iStateVars=row['iStateVars'],
                               parameters=row['parameters'],
                               outputs=self.outputs,
                               model_functions=self.model_functions)
            self.add_cow(cow_variable)
        print("All cows have been added")
        return
    
    def get_cow_by_id(self, cow_id):
        for cow in self.list_of_cows:
            if cow.name == cow_id:
                print("Cow found")
                return cow
        print("Cow not found")
        return 
   

    def run_phase_feed(self, scenario_index):
        # scenario_index = index of scenario in the list
        if scenario_index < len(self.scenario_list):
            selected_scenario = self.scenario_list[scenario_index]

        for cow in self.list_of_cows:
            prev_output = None  # Set it to the appropriate DataFrame if available
            selected_scenario.run_scenario(cow, prev_output=prev_output)
            prev_output = cow.results  # Update prev_output for the next scenario


            # for scenario_index in range(scenario_selection + 1):  # Run scenarios up to the selected one
                # selected_scenario = self.scenario_list[scenario_index]
                # selected_scenario.run_scenario(cow, prev_output=prev_output)
                # prev_output = cow.results  # Update prev_output for the next scenario


            # for cow in self.list_of_cows:
            #     prev_output = None  # Set it to the appropriate DataFrame if available
            #     selected_scenario.run_scenario(cow, prev_output=prev_output)
            #     prev_output = cow.results  # Update prev_output for the next cow



In [9]:
# Define model
def laccurve5_dijkstra(self, stateVars, t, scenario_param=None, prev_var_return=None):
    # prev_var_return is required if you want to have a variable with an initial value that then gets updated each iteration
    # The variable must also be included in the outputs
    # TODO ask John about this issue, how common is it to have variables like this

    import numpy as np
    import math

    # Assign Parameter Values #
    a = self.parameters['a']
    b = self.parameters['b']
    b0 = self.parameters['b0']
    c = self.parameters['c']
    L = self.parameters['L']
    KDMIMILK = self.parameters['KDMIMILK']
    KDMImbw = self.parameters['KDMImbw']
    # Hcfeed = self.parameters['Hcfeed']
                # Hcfeed is replaced with scenario_param
    Hcmilk = self.parameters['Hcmilk']
    KL = self.parameters['KL']
    expL = self.parameters['expL']
    pregdate = self.parameters['pregdate']
    kf1 = self.parameters['kf1']
    kf2 = self.parameters['kf2']
    milk_value = self.parameters['milk_value']

    # Paramaters that require an initial value #
    if t == 0:
            E = self.parameters['E']
    else:
        E = prev_var_return[5]


    # Variables w/ Differential Equation #
    BW = stateVars[0]
    ER = stateVars[1]
    milkcumul = stateVars[2]
    DMIcumul = stateVars[3]
    IOFCcumul = stateVars[4]

    # Model Equations # 
    feed_price = (101 * scenario_param + 2.7) / 1000

    BFat = ER/9.0       
    # Body fat

    Lmod = 1.0-(1.0-L)/(1.0+(KL/BFat)**expL)         
        # 1 - (1-L) = L?
        # Modifying value of L? Why the adjustment?

    dijkstra_milk = a * np.exp(b * (1 - np.exp(-b0 * t)) / b0 - c * t)
    # John has modified S to give 4% FCM yield per alveolus 

    NEmilk = E**Lmod*dijkstra_milk                         
        # Net energy milk
        # On slide 28 this is equation for FCM yield but L is modified here
        # Is L modified to give a NEmilk as well as a milk yield (milk)

    milk = NEmilk/Hcmilk                
        # Milk production

    DMINRC = (KDMIMILK * milk + KDMImbw * BW**0.75)*(1.0-math.exp(-0.192*(t/7+3.67)))   
        # Slide 28
        # NRC DMI equation

    fdbk = (kf1*t/7+kf2)*(ER-self.iStateVars[1])/self.iStateVars[1]       
    # Ellis 2006
        # Feedback on DMI

    DMI = DMINRC - fdbk                             
        # Dry matter intake     

    NEI = scenario_param*DMI                                
        # Net energy intake

    NEmaint = 0.096*BW**0.75                        
        # Net energy maintenance

    if t < pregdate + 190:                          
        NEgest = 0
    else:
        NEgest = (0.00318*(t-pregdate-190)-0.0352)/0.218
        # Net energy gestation

    NEreqt = NEmaint + dijkstra_milk + NEgest + 5.12*2.0          
        # NE requirement, from NRC

    E = NEI/NEreqt            
        # Energy balance

    # Differential Equations # 
    dERdt = NEI - NEmaint - NEmilk - NEgest
    # Energy requirement (NE)

    if dERdt < 0:
        dBWdt = dERdt/4.92  # 4.92 Mcal/kg in NRC(1988)
    else:
        dBWdt = dERdt/5.12  # 5.12 Mcal/kg for gain
        # Bodyweight gain

    IOFC = (milk * milk_value) - (DMI * feed_price)

    # Format data for return # 
    differential_return = [dBWdt, dERdt, milk, DMI, IOFC]
    local_variables = locals()
    # Store local variables 
    variable_returns = [local_variables.get(variable_name) for variable_name in self.outputs]
    # Create list of variables to return

    return differential_return, variable_returns

functions_to_include = [laccurve5_dijkstra]

In [10]:
# TEST #
import pandas as pd

# 1. Create scenario
parameters_list = [1.7, 1.3]
scenario_phase_feed = Scenario(parameters_list=parameters_list, scenario_function=phase_feed)

# 2. Create herd
output = ['t', 'milk', 'DMI', 'E', 'BW', 'BFat', 'ER', 'IOFC', 'milkcumul', 'DMIcumul', 'IOFCcumul'] # Model outputs
scenario_list=[scenario_phase_feed]
herd = Herd(outputs=output, model_functions=functions_to_include, scenario_list=scenario_list)

# 3. Add cows
cow_data_raw = pd.read_csv('../data/cow_import_data.csv')
# Function to convert a .csv to the required format
def cow_wrangler(df):
    # Initialize empty lists to store data for each cow
    cow_ids = []
    iStateVars_list = []
    parameters_dict_list = []

    # Iterate over rows
    for index, row in df.iterrows():
        # Extract cow ID
        cow_id = row['cow_ID']
        cow_ids.append(cow_id)

        # Extract iStateVars
        iStateVars = [row[col] for col in df.columns if col.startswith('iStateVar_')]
        iStateVars_list.append(iStateVars)

        # Extract parameters
        parameters_dict = {col.replace('param_', ''): row[col] for col in df.columns if col.startswith('param_')}
        parameters_dict_list.append(parameters_dict)

    # Create a new DataFrame
    wrangled_data = pd.DataFrame({
        'cow_ID': cow_ids,
        'iStateVars': iStateVars_list,
        'parameters': parameters_dict_list
    })

    return wrangled_data
cow_data_wranlged = cow_wrangler(cow_data_raw)

herd.import_dataframe(df=cow_data_wranlged)

# 4. Run scenario
herd.run_phase_feed(0)

All cows have been added
Running Model....
       t        milk        DMI         E          BW        BFat  \
0   0.00   12.754638   8.701276  0.402500  650.000000  117.000000   
1   6.99  198.945167  49.937885  2.171139  507.745946   39.234451   
2  13.99    8.748258   8.289452  0.336392  436.507061    0.290527   
3  20.99    7.074556   8.573388  0.323726  436.350741    0.206913   
4  27.99   10.845753  10.232772  0.367191  436.497705    0.290519   
5  34.99   14.682609  11.923803  0.413457  436.639832    0.371374   
6  41.99   18.522694  13.617448  0.462077  436.781829    0.452155   
7  48.99   22.302394  15.284672  0.512324  436.925153    0.533690   

            ER        IOFC    milkcumul     DMIcumul    IOFCcumul  
0  1053.000000    9.961672     0.000000     0.000000     0.000000  
1   353.110056  170.341483  1727.159796   395.553984  1485.459201  
2     2.614739    6.427752  2721.066404   666.610711  2332.702856  
3     1.862218    4.871901  2762.594951   722.202105  2360.3834

UnboundLocalError: cannot access local variable 'variable_returns' where it is not associated with a value